# Reinforcement Learning & Diffusion Models
## Part 1: Value Iteration on FrozenLake MDP | Part 2: Diffusion Model Mini-Research

---

**Course:** Decentralized AI  
**References:**
- Sutton & Barto, *Reinforcement Learning: An Introduction* (2018)
- Ho et al., *Denoising Diffusion Probabilistic Models* (NeurIPS 2020) — https://arxiv.org/abs/2006.11239
- Song et al., *Denoising Diffusion Implicit Models* (ICLR 2021) — https://arxiv.org/abs/2010.02502
- Nichol & Dhariwal, *Improved Denoising Diffusion Probabilistic Models* (ICML 2021) — https://arxiv.org/abs/2102.09672
- OpenAI Gymnasium docs — https://gymnasium.farama.org/
- HuggingFace Diffusers docs — https://huggingface.co/docs/diffusers

---

# Part 1: Reinforcement Learning (40 pts)
## Task 1 — Value Iteration on FrozenLake MDP

### MDP Definition

**Environment:** `FrozenLake-v1` (4×4 grid, non-slippery variant for clarity, then slippery for realism)

```
S F F F
F H F H
F F F H
H F F G
```
- `S` = Start, `F` = Frozen (safe), `H` = Hole (terminal, reward = 0), `G` = Goal (reward = +1)

**State space:** S = {0, 1, ..., 15} — one integer per grid cell (row-major order)

**Action space:** A = {0=Left, 1=Down, 2=Right, 3=Up}

**Reward structure:**
- Reach goal (state 15): R = +1
- Fall in hole or step on frozen tile: R = 0

**Discount factor γ:** default = 0.99 (we will vary this in the experiment)

**Why is this an MDP?**  
FrozenLake satisfies the Markov property: the next state and reward depend *only* on the current state and action, not on history. The transition model P(s'|s,a) is fully defined by the environment's dynamics (including stochastic slippage on ice). The agent has no memory requirement — only the current grid cell matters for decision-making.

In [ ]:
!pip install gymnasium --quiet
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import warnings
warnings.filterwarnings('ignore')
print('Libraries loaded ✓')

### 1.1 Value Iteration Implementation

In [ ]:
def value_iteration(env, gamma=0.99, theta=1e-8, max_iter=10_000):
    n_states  = env.observation_space.n
    n_actions = env.action_space.n
    P = env.unwrapped.P#unwrap to access transition table
    V = np.zeros(n_states)
    deltas = []
    for _ in range(max_iter):
        delta = 0.0
        for s in range(n_states):
            v_old = V[s]
            action_values = np.zeros(n_actions)
            for a in range(n_actions):
                for prob, next_s, reward, done in P[s][a]:
                    action_values[a] += prob * (reward + gamma * V[next_s] * (not done))
            V[s] = np.max(action_values)
            delta = max(delta, abs(v_old - V[s]))
        deltas.append(delta)
        if delta < theta:
            break
    policy = np.zeros(n_states, dtype=int)
    for s in range(n_states):
        action_values = np.zeros(n_actions)
        for a in range(n_actions):
            for prob, next_s, reward, done in P[s][a]:
                action_values[a] += prob * (reward + gamma * V[next_s] * (not done))
        policy[s] = np.argmax(action_values)
    return V, policy, deltas

def evaluate_policy(env, policy, n_episodes=2000):
    successes = 0
    for _ in range(n_episodes):
        s, _ = env.reset()
        done = False
        while not done:
            s, r, terminated, truncated, _ = env.step(policy[s])
            done = terminated or truncated
        successes += (r == 1.0)
    return successes / n_episodes

print('value_iteration() defined ✓')

### 1.2 Run Value Iteration (γ = 0.99, slippery FrozenLake)

In [ ]:
#(is_slippery=True = stochastic, more realistic MDP)
env = gym.make('FrozenLake-v1', is_slippery=True)
env_eval = gym.make('FrozenLake-v1', is_slippery=True)
V, policy, deltas = value_iteration(env, gamma=0.99)
print(f'Converged in {len(deltas)} iterations')
print(f'\nOptimal Value Function V* (reshaped to 4×4 grid):')
print(np.round(V.reshape(4, 4), 4))
ACTION_NAMES = {0: '←', 1: '↓', 2: '→', 3: '↑'}
policy_display = np.array([ACTION_NAMES[a] for a in policy]).reshape(4, 4)
print(f'\nOptimal Policy π* (4×4 grid):')
print(policy_display)
success_rate = evaluate_policy(env_eval, policy, n_episodes=5000)
print(f'\nPolicy success rate (5000 episodes): {success_rate:.2%}')

### 1.3 Visualize Value Function and Policy

In [ ]:
GRID = ['S','F','F','F',
        'F','H','F','H',
        'F','F','F','H',
        'H','F','F','G']
HOLES = {i for i,c in enumerate(GRID) if c == 'H'}
GOAL  = {15}
fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
fig.patch.set_facecolor('#0f0f14')
for ax in axes:
    ax.set_facecolor('#0f0f14')
ax = axes[0]#Left: Value Function heatmap
cmap = LinearSegmentedColormap.from_list('frozen', ['#1a1a2e','#16213e','#0f3460','#53d8fb','#ffffff'])
V_grid = V.reshape(4, 4)
im = ax.imshow(V_grid, cmap=cmap, vmin=0, vmax=V.max())
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04).ax.tick_params(colors='white')
for i in range(4):
    for j in range(4):
        idx = i*4+j
        cell = GRID[idx]
        val  = V[idx]
        color = '#ff4757' if cell=='H' else ('#2ed573' if cell=='G' else 'white')
        ax.text(j, i, f'{val:.3f}', ha='center', va='center', fontsize=11, color=color, fontweight='bold', fontfamily='monospace')
        ax.text(j, i-0.33, cell, ha='center', va='center', fontsize=8, color='#aaaaaa', fontfamily='monospace')
ax.set_title('Optimal Value Function V* (γ=0.99)', color='white', fontsize=13, pad=12)
ax.set_xticks([]); ax.set_yticks([])
for spine in ax.spines.values(): spine.set_edgecolor('#333355')
#Right: Policy arrows
ax = axes[1]
bg = np.zeros((4,4))
for idx in HOLES:  bg[idx//4, idx%4] = -1
for idx in GOAL:   bg[idx//4, idx%4] =  1
cmap2 = LinearSegmentedColormap.from_list('pg', ['#ff4757','#1e2337','#2ed573'])
ax.imshow(bg, cmap=cmap2, vmin=-1, vmax=1, alpha=0.65)
dx = {0:-0.35, 1:0, 2:0.35, 3:0}
dy = {0:0, 1:0.35, 2:0, 3:-0.35}
for i in range(4):
    for j in range(4):
        idx = i*4+j
        cell = GRID[idx]
        if cell in ('H','G'):
            sym = '✕' if cell=='H' else '★'
            ax.text(j, i, sym, ha='center', va='center', fontsize=18, color='white' if cell=='G' else '#ff6b81')
        else:
            a = policy[idx]
            ax.annotate('', xy=(j+dx[a], i+dy[a]), xytext=(j, i), arrowprops=dict(arrowstyle='->', color='#53d8fb', lw=2.0))
ax.set_xlim(-0.5, 3.5); ax.set_ylim(3.5, -0.5)
ax.set_xticks(range(4)); ax.set_yticks(range(4))
ax.set_xticklabels(range(4), color='#aaaaaa')
ax.set_yticklabels(range(4), color='#aaaaaa')
ax.grid(True, color='#333355', linewidth=0.8)
ax.set_title('Optimal Policy π* (arrows = best action)', color='white', fontsize=13, pad=12)
legend_items = [mpatches.Patch(color='#2ed573', label='Goal'), mpatches.Patch(color='#ff4757', label='Hole'), mpatches.Patch(color='#1e2337', label='Frozen')]
ax.legend(handles=legend_items, loc='lower right', framealpha=0.3, labelcolor='white', facecolor='#1e2337', edgecolor='#333355')
plt.suptitle('FrozenLake-v1 · Value Iteration Results', color='white', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('vi_results.png', dpi=150, bbox_inches='tight', facecolor='#0f0f14')
plt.show()
print('Figure saved: vi_results.png')

### 1.4 Convergence of Value Iteration

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
fig.patch.set_facecolor('#0f0f14')
ax.set_facecolor('#0f0f14')
ax.semilogy(deltas, color='#53d8fb', linewidth=2)
ax.axhline(1e-8, color='#ff4757', linestyle='--', linewidth=1, label='θ = 1e-8')
ax.set_xlabel('Iteration', color='white'); ax.set_ylabel('Max |ΔV|', color='white')
ax.set_title('Value Iteration Convergence (γ=0.99)', color='white', fontsize=13)
ax.tick_params(colors='white'); ax.legend(labelcolor='white', facecolor='#1e2337', edgecolor='#333355')
for spine in ax.spines.values(): spine.set_edgecolor('#333355')
ax.grid(True, color='#1e2337', linewidth=0.7)
plt.tight_layout()
plt.savefig('vi_convergence.png', dpi=150, bbox_inches='tight', facecolor='#0f0f14')
plt.show()
print(f'Total iterations to convergence: {len(deltas)}')

### 1.5 Experiment: How does γ (discount factor) affect the policy?

**Hypothesis:** A low γ makes the agent short-sighted — it may prefer nearby rewards (or avoid risk by staying still) rather than navigating toward the distant goal. A high γ encourages long-term planning, yielding a more direct path.

In [ ]:
gammas = [0.50, 0.70, 0.90, 0.95, 0.99]
results = {}
for g in gammas:
    env_g = gym.make('FrozenLake-v1', is_slippery=True)
    V_g, pi_g, d_g = value_iteration(env_g, gamma=g)
    sr = evaluate_policy(gym.make('FrozenLake-v1', is_slippery=True), pi_g, n_episodes=3000)
    results[g] = {'V': V_g, 'policy': pi_g, 'iters': len(d_g), 'success': sr}
    print(f'γ={g:.2f}  →  converged in {len(d_g):4d} iters  |  success rate: {sr:.2%}')

In [ ]:
#Side-by-side policy grids for all γ values
fig, axes = plt.subplots(1, len(gammas), figsize=(18, 4))
fig.patch.set_facecolor('#0f0f14')
for ax, g in zip(axes, gammas):
    ax.set_facecolor('#0f0f14')
    bg = np.zeros((4,4))
    for idx in HOLES: bg[idx//4, idx%4] = -1
    for idx in GOAL:  bg[idx//4, idx%4] =  1
    ax.imshow(bg, cmap=cmap2, vmin=-1, vmax=1, alpha=0.55)
    pi_g = results[g]['policy']
    for i in range(4):
        for j in range(4):
            idx = i*4+j
            cell = GRID[idx]
            if cell in ('H','G'):
                sym = '✕' if cell=='H' else '★'
                ax.text(j, i, sym, ha='center', va='center', fontsize=16, color='white' if cell=='G' else '#ff6b81')
            else:
                a = pi_g[idx]
                ax.annotate('', xy=(j+dx[a], i+dy[a]), xytext=(j, i), arrowprops=dict(arrowstyle='->', color='#53d8fb', lw=2.0))
    ax.set_xlim(-0.5,3.5); ax.set_ylim(3.5,-0.5)
    ax.set_xticks([]); ax.set_yticks([])
    ax.grid(True, color='#333355', linewidth=0.7)
    sr = results[g]['success']
    ax.set_title(f'γ = {g}\n({sr:.0%} success)', color='white', fontsize=11)
plt.suptitle('Policy Changes Across Discount Factors γ', color='white', fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig('gamma_experiment.png', dpi=150, bbox_inches='tight', facecolor='#0f0f14')
plt.show()

In [ ]:
#Success rate vs γ bar chart
fig, ax = plt.subplots(figsize=(8, 4))
fig.patch.set_facecolor('#0f0f14'); ax.set_facecolor('#0f0f14')
rates = [results[g]['success'] for g in gammas]
bars  = ax.bar([str(g) for g in gammas], rates, color=['#ff4757','#ff6348','#ffa502','#53d8fb','#2ed573'])
for bar, r in zip(bars, rates): ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005, f'{r:.1%}', ha='center', va='bottom', color='white', fontsize=10, fontweight='bold')
ax.set_xlabel('Discount Factor γ', color='white')
ax.set_ylabel('Success Rate', color='white')
ax.set_title('Policy Success Rate vs. Discount Factor γ', color='white', fontsize=13)
ax.set_ylim(0, max(rates)+0.08)
ax.tick_params(colors='white')
for spine in ax.spines.values(): spine.set_edgecolor('#333355')
ax.grid(True, axis='y', color='#1e2337', linewidth=0.7)
plt.tight_layout()
plt.savefig('success_vs_gamma.png', dpi=150, bbox_inches='tight', facecolor='#0f0f14')
plt.show()

### 1.6 Discussion

**Why is FrozenLake an MDP?**  
FrozenLake satisfies all MDP axioms: (1) a finite, fully-enumerable state space S; (2) a discrete action space A; (3) a stationary, Markovian transition kernel P(s′|s,a) — the next cell and reward depend only on the current cell and move, not on history; (4) a bounded reward function R(s,a,s′). The stochastic "slippery" variant adds a richer transition distribution (the agent slips perpendicular to intended direction with probability 2/3), which makes optimal planning non-trivial.

**What is the optimal policy doing?**  
The policy found by value iteration is a *risk-aware* path to the goal. On the slippery surface, the agent learns to hug the bottom and right edges — states farther from holes — rather than cutting directly through the center. The arrows in columns adjacent to holes consistently point *away* from the hole, even when that means a longer route. This is optimal because slipping into a hole yields R=0 and terminates the episode.

**Effect of γ on the policy:**  
| γ    | Behavior                                                                                  |
|------|-------------------------------------------------------------------------------------------|
| 0.50 | Very myopic. Some states collapse to degenerate actions since distant reward is discounted to near-zero. Success rate drops significantly. |
| 0.70 | Slightly better, but still fails to plan around holes reliably.                          |
| 0.90 | Mostly correct policy emerges; success rate climbs.                                      |
| 0.99 | Full long-horizon planning. Highest success rate. Policy carefully avoids holes even from distance. |

As γ increases, the value function assigns more weight to future rewards. Since the only non-zero reward is at the goal (16 steps max away), low γ effectively destroys the signal. High γ preserves it, enabling the agent to navigate safely across the board.

---
# Part 2: Mini-Research — Denoising Diffusion Probabilistic Models (DDPMs) (60 pts)

## 2.1 Background and Problem Statement

### What problem do diffusion models solve?

Generative modeling asks: *given a dataset of images (or audio, video, etc.), learn a distribution from which we can sample new examples that are statistically indistinguishable from real data.* Prior approaches (GANs, VAEs, normalizing flows) each carry tradeoffs — GANs suffer from training instability and mode collapse; VAEs produce blurry samples; flows impose architectural constraints for invertibility.

**Denoising Diffusion Probabilistic Models (DDPMs)** (Ho et al., NeurIPS 2020) address this by reformulating generation as a *learned denoising process*: instead of directly learning to hallucinate images, the model learns to *reverse* a known Gaussian noise-adding process. This yields sample quality competitive with or exceeding GANs while offering stable training and exact log-likelihood bounds.

### Primary Sources
1. Ho, J., Jain, A., & Abbeel, P. (2020). *Denoising Diffusion Probabilistic Models.* NeurIPS. https://arxiv.org/abs/2006.11239  
2. Song, J., Meng, C., & Ermon, S. (2021). *Denoising Diffusion Implicit Models.* ICLR. https://arxiv.org/abs/2010.02502  
3. HuggingFace Diffusers library — https://huggingface.co/docs/diffusers (official documentation / model cards)

---

## 2.2 Core Idea: The Forward and Reverse Process

### Forward Process (Fixed)
Given a clean image **x₀**, we define a Markov chain of T steps that *gradually adds* Gaussian noise:

$$q(\mathbf{x}_t | \mathbf{x}_{t-1}) = \mathcal{N}(\mathbf{x}_t; \sqrt{1-\beta_t}\,\mathbf{x}_{t-1},\; \beta_t \mathbf{I})$$

where β₁, ..., β_T is a **noise schedule** (small positive constants). After T steps (typically T=1000), x_T ≈ 𝒩(0, I) — pure Gaussian noise. Crucially, we can sample x_t at *any* step directly:

$$q(\mathbf{x}_t | \mathbf{x}_0) = \mathcal{N}(\mathbf{x}_t;\; \sqrt{\bar\alpha_t}\,\mathbf{x}_0,\; (1-\bar\alpha_t)\mathbf{I})$$

where $\bar\alpha_t = \prod_{s=1}^t (1-\beta_s)$. This allows efficient training.

### Reverse Process (Learned)
The reverse process — denoising x_T back to x_0 — is also Markovian:

$$p_\theta(\mathbf{x}_{t-1} | \mathbf{x}_t) = \mathcal{N}(\mathbf{x}_{t-1};\; \mu_\theta(\mathbf{x}_t, t),\; \Sigma_\theta(\mathbf{x}_t, t))$$

A neural network (typically a **U-Net with time conditioning**) learns to predict the noise ε that was added at each step, from which we recover the mean μ_θ.

### Training Objective
Ho et al. simplify the variational lower bound to:

$$\mathcal{L}_{\text{simple}} = \mathbb{E}_{t, \mathbf{x}_0, \boldsymbol{\varepsilon}} \left[ \left\| \boldsymbol{\varepsilon} - \boldsymbol{\varepsilon}_\theta(\sqrt{\bar\alpha_t}\,\mathbf{x}_0 + \sqrt{1-\bar\alpha_t}\,\boldsymbol{\varepsilon},\; t) \right\|^2 \right]$$

This is simply **MSE between the true noise ε and the predicted noise ε_θ** — a clean, stable regression objective.

### DDIM (Song et al., 2021) — Faster Sampling
DDPM requires T=1000 steps for sampling. DDIM reformulates sampling as a non-Markovian process, enabling high-quality generation in as few as 10–50 steps — dramatically reducing inference time with minimal quality loss.

## 2.3 Hands-On Component: Forward Diffusion Visualization + Scheduler Comparison

We perform two hands-on experiments:
1. **Visualize the forward diffusion process** on a real image to build intuition for the noise schedule
2. **Compare DDPM vs DDIM schedulers** — quantitatively and visually — on a pretrained unconditional diffusion model, varying the number of denoising steps

In [ ]:
#Install diffusers stack
!pip install diffusers transformers accelerate Pillow --quiet
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from diffusers import DDPMScheduler, DDIMScheduler, DDPMPipeline
import requests, io, time
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

### Experiment A: Forward Diffusion Process Visualization

In [ ]:
url = 'https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/cat.png'
response = requests.get(url, timeout=15)
img = Image.open(io.BytesIO(response.content)).convert('RGB').resize((256, 256))
x0 = torch.tensor(np.array(img)).permute(2,0,1).float() / 127.5 - 1.0  # [-1,1]
x0 = x0.unsqueeze(0)  # [1, 3, 256, 256]
#DDPM noise schedule (linear beta schedule)
T = 1000
scheduler = DDPMScheduler(num_train_timesteps=T, beta_schedule='linear')
alphas_cumprod = scheduler.alphas_cumprod  # ᾱ_t for all t
#Visualize forward process at selected timesteps
timesteps_to_show = [0, 50, 100, 200, 400, 600, 800, 999]
def noisy_image_at_t(x0, t, scheduler):
    """Sample x_t from q(x_t|x_0) using the reparameterization."""
    noise = torch.randn_like(x0)
    alpha_bar = scheduler.alphas_cumprod[t]
    x_t = (alpha_bar**0.5) * x0 + ((1 - alpha_bar)**0.5) * noise
    return x_t

def to_pil(tensor):
    arr = tensor.squeeze().permute(1,2,0).numpy()
    arr = np.clip((arr + 1) / 2.0 * 255, 0, 255).astype(np.uint8)
    return Image.fromarray(arr)
fig, axes = plt.subplots(1, len(timesteps_to_show), figsize=(20, 3.5))
fig.patch.set_facecolor('#0f0f14')
for ax, t in zip(axes, timesteps_to_show):
    ax.set_facecolor('#0f0f14')
    if t == 0:
        pil_img = img
    else:
        noisy = noisy_image_at_t(x0, t, scheduler)
        pil_img = to_pil(noisy)
    ax.imshow(pil_img)
    ax.set_title(f't = {t}\nᾱ_t = {scheduler.alphas_cumprod[t]:.3f}', color='white', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values(): spine.set_edgecolor('#333355')
plt.suptitle('Forward Diffusion Process: Gradual Noise Addition (Linear β Schedule)', color='white', fontsize=13, fontweight='bold', y=1.05)
plt.tight_layout()
plt.savefig('forward_diffusion.png', dpi=150, bbox_inches='tight', facecolor='#0f0f14')
plt.show()

In [ ]:
#Plot αᾱ_t and signal-to-noise ratio over time
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.patch.set_facecolor('#0f0f14')
ts = np.arange(T)
ac = scheduler.alphas_cumprod.numpy()
for ax in axes: ax.set_facecolor('#0f0f14')
axes[0].plot(ts, ac, color='#53d8fb', lw=2, label='ᾱ_t (signal power)')
axes[0].plot(ts, 1 - ac, color='#ff4757', lw=2, label='1 − ᾱ_t (noise power)')
axes[0].set_xlabel('Timestep t', color='white')
axes[0].set_ylabel('Value', color='white')
axes[0].set_title('Signal vs. Noise Power (Linear Schedule)', color='white', fontsize=11)
axes[0].tick_params(colors='white')
axes[0].legend(labelcolor='white', facecolor='#1e2337', edgecolor='#333355')
axes[0].grid(True, color='#1e2337')
for sp in axes[0].spines.values(): sp.set_edgecolor('#333355')
snr = ac / (1 - ac + 1e-8)
axes[1].semilogy(ts, snr, color='#2ed573', lw=2)
axes[1].set_xlabel('Timestep t', color='white')
axes[1].set_ylabel('SNR = ᾱ_t / (1 − ᾱ_t)', color='white')
axes[1].set_title('Signal-to-Noise Ratio vs. Timestep', color='white', fontsize=11)
axes[1].tick_params(colors='white')
axes[1].grid(True, color='#1e2337')
for sp in axes[1].spines.values(): sp.set_edgecolor('#333355')
plt.suptitle('Noise Schedule Analysis', color='white', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('noise_schedule.png', dpi=150, bbox_inches='tight', facecolor='#0f0f14')
plt.show()

### Experiment B: DDPM vs DDIM Scheduler Comparison — Varying Denoising Steps

We load `google/ddpm-celebahq-256`, a pretrained unconditional diffusion model, and compare:
- **DDPM** scheduler: Markovian, requires many steps
- **DDIM** scheduler: Non-Markovian, deterministic, fewer steps needed

We measure: inference time, and visually compare sample quality at 10, 25, 50, and 100 steps.

In [ ]:
from diffusers import UNet2DModel
#Load pretrained U-Net backbone (weights only, reuse for both schedulers)
model_id = 'google/ddpm-celebahq-256'
unet = UNet2DModel.from_pretrained(model_id).to(device)
unet.eval()
print('U-Net loaded ✓')
print(f'  Parameters: {sum(p.numel() for p in unet.parameters())/1e6:.1f}M')

In [ ]:
@torch.no_grad()
def sample_images(unet, scheduler, num_inference_steps, num_images=4, seed=42):
    """
    Generate images using a given scheduler and number of steps.
    Returns: PIL images, elapsed time (seconds)
    """
    scheduler.set_timesteps(num_inference_steps)
    generator = torch.manual_seed(seed)
    #Start from pure Gaussian noise
    sample = torch.randn((num_images, 3, 256, 256),generator=generator, device=device)
    start = time.time()
    for t in scheduler.timesteps:
        #Predict noise residual
        t_batch = torch.full((num_images,), t, device=device, dtype=torch.long)
        noise_pred = unet(sample, t_batch).sample
        #Scheduler step (reverse diffusion)
        sample = scheduler.step(noise_pred, t, sample).prev_sample
    elapsed = time.time() - start
    #Convert to PIL
    images = (sample / 2 + 0.5).clamp(0, 1)
    images = images.cpu().permute(0, 2, 3, 1).numpy()
    pil_images = [Image.fromarray((img * 255).astype(np.uint8)) for img in images]
    return pil_images, elapsed
print('sample_images() defined ✓')

In [ ]:
step_counts = [10, 25, 50, 100]
timing_results = {'DDPM': {}, 'DDIM': {}}
image_results  = {'DDPM': {}, 'DDIM': {}}
ddpm_sched = DDPMScheduler.from_pretrained(model_id)
ddim_sched = DDIMScheduler.from_pretrained(model_id)
for steps in step_counts:
    print(f'\nRunning {steps} steps...')
    #DDPM
    imgs_ddpm, t_ddpm = sample_images(unet, ddpm_sched, steps, num_images=1)
    image_results['DDPM'][steps]  = imgs_ddpm[0]
    timing_results['DDPM'][steps] = t_ddpm
    print(f'  DDPM  {steps:3d} steps: {t_ddpm:.2f}s')
    #DDIM
    imgs_ddim, t_ddim = sample_images(unet, ddim_sched, steps, num_images=1)
    image_results['DDIM'][steps]  = imgs_ddim[0]
    timing_results['DDIM'][steps] = t_ddim
    print(f'  DDIM  {steps:3d} steps: {t_ddim:.2f}s')

In [ ]:
#Side-by-side visual comparison
fig, axes = plt.subplots(2, len(step_counts), figsize=(16, 9))
fig.patch.set_facecolor('#0f0f14')
for col, steps in enumerate(step_counts):
    for row, sched_name in enumerate(['DDPM', 'DDIM']):
        ax = axes[row, col]
        ax.imshow(image_results[sched_name][steps])
        t = timing_results[sched_name][steps]
        ax.set_title(f'{steps} steps\n{t:.1f}s', color='white', fontsize=10)
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values(): spine.set_edgecolor('#53d8fb' if sched_name=='DDIM' else '#ff4757')
        spine_lw = 2
        for spine in ax.spines.values(): spine.set_linewidth(spine_lw)
    axes[0, col].set_title(f'{step_counts[col]} steps\n{timing_results["DDPM"][steps]:.1f}s', color='white', fontsize=10)
axes[0, 0].set_ylabel('DDPM', color='#ff4757', fontsize=13, fontweight='bold')
axes[1, 0].set_ylabel('DDIM', color='#53d8fb', fontsize=13, fontweight='bold')
plt.suptitle('DDPM vs DDIM — Generated Faces at Different Step Counts\n(google/ddpm-celebahq-256, same random seed)', color='white', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('scheduler_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0f0f14')
plt.show()

In [ ]:
#Timing comparison table + bar chart
import pandas as pd
rows = []
for steps in step_counts:
    rows.append({
        'Steps': steps,
        'DDPM Time (s)': round(timing_results['DDPM'][steps], 2),
        'DDIM Time (s)': round(timing_results['DDIM'][steps], 2),
        'DDIM Speedup': round(timing_results['DDPM'][steps]/timing_results['DDIM'][steps], 2)
    })
df = pd.DataFrame(rows)
print('Timing Comparison Table')
print('=' * 55)
print(df.to_string(index=False))
fig, ax = plt.subplots(figsize=(9, 4.5))
fig.patch.set_facecolor('#0f0f14'); ax.set_facecolor('#0f0f14')
x = np.arange(len(step_counts)); w = 0.35
b1 = ax.bar(x - w/2, [timing_results['DDPM'][s] for s in step_counts], w, label='DDPM', color='#ff4757', alpha=0.85)
b2 = ax.bar(x + w/2, [timing_results['DDIM'][s] for s in step_counts], w, label='DDIM', color='#53d8fb', alpha=0.85)
for bar in list(b1) + list(b2): ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3, f'{bar.get_height():.1f}s', ha='center', va='bottom', color='white', fontsize=9, fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels([f'{s} steps' for s in step_counts], color='white')
ax.set_ylabel('Inference Time (s)', color='white')
ax.set_title('DDPM vs DDIM Inference Time by Step Count', color='white', fontsize=12)
ax.tick_params(colors='white')
ax.legend(labelcolor='white', facecolor='#1e2337', edgecolor='#333355')
ax.grid(True, axis='y', color='#1e2337')
for sp in ax.spines.values(): sp.set_edgecolor('#333355')
plt.tight_layout()
plt.savefig('timing_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0f0f14')
plt.show()

## 2.4 Results Summary Table

| Scheduler | Steps | Time (s) | Qualitative Quality |
|-----------|-------|----------|--------------------|
| DDPM      | 10    | —        | Noisy/blurry; key facial features missing |
| DDPM      | 25    | —        | Somewhat recognizable faces; artifacts present |
| DDPM      | 50    | —        | Decent quality; minor blur |
| DDPM      | 100   | —        | Good quality; approaching full-step quality |
| DDIM      | 10    | —        | Reasonably sharp; coherent structure |
| DDIM      | 25    | —        | Clear faces; close to 100-step DDPM quality |
| DDIM      | 50    | —        | High quality; minimal artifacts |
| DDIM      | 100   | —        | Near-optimal quality |

*Note: Exact timing numbers will fill in after running the cell above. DDIM consistently outperforms DDPM at low step counts because its non-Markovian formulation allows larger effective stride per step without breaking consistency.*

## 2.5 Architecture: The U-Net with Time Conditioning

The backbone network ε_θ(x_t, t) is a **U-Net** adapted from image segmentation, augmented with:

1. **Sinusoidal time embeddings:** The timestep t is encoded as a sinusoidal embedding (similar to transformer positional encodings), then projected through a small MLP. This embedding is *added* into each residual block, giving the model knowledge of how noisy the input currently is.

2. **Residual blocks:** Each block applies GroupNorm → SiLU activation → Convolution, plus a shortcut connection, with the time embedding injected via feature-wise affine transformation.

3. **Self-attention layers:** Inserted at low spatial resolutions (16×16, 8×8) to capture long-range dependencies — important for coherent global structure in faces.

4. **Skip connections (U-Net structure):** The encoder-decoder design with skip connections allows high-frequency spatial details from early layers to be recovered during decoding — critical for sharp textures.

The model is trained entirely on the noise-prediction objective L_simple described above — no discriminator, no KL divergence computation per pixel, just straightforward MSE.

## 2.6 Limitations, Risks, and Open Challenges

### Limitation 1: Slow Inference
DDPMs require T=1000 sequential neural network forward passes at full resolution. Even with DDIM reducing this to 50 steps, generating a single 256×256 image takes several seconds on a GPU — orders of magnitude slower than a GAN's single forward pass. This limits real-time applications (interactive tools, video generation). While consistency models (Song et al., 2023) partially address this, the fundamental sequential denoising architecture remains a bottleneck.

### Limitation 2: Memorization and Privacy Risks
Carlini et al. (2023) demonstrated that diffusion models trained on large-scale web-scraped datasets can *memorize and regenerate* near-verbatim training images. This raises serious concerns: (a) copyright infringement when training data includes copyrighted artwork; (b) privacy violations if the training set contains personal photographs; (c) lack of recourse for individuals whose data was used. Unlike discriminative models where memorization is a localized phenomenon, diffusion model memorization can be triggered by specific prompts, making it harder to audit at scale.

### Open Challenge: Semantic Controllability
While text-conditioned diffusion (e.g., Stable Diffusion) offers impressive control, fine-grained manipulation — changing exactly one attribute (e.g., hair color) while preserving everything else — remains unreliable. Current approaches like prompt editing, SDEdit, and DreamBooth offer partial solutions but lack formal guarantees on preserving unspecified attributes. This is an active area of research.

## 2.7 Proposed Extension

**Experiment: Adaptive Noise Schedules for Rare-Feature Preservation**

Standard linear or cosine noise schedules treat all image regions equally. However, rare visual features (fine textures, unusual colors, small objects) tend to be destroyed at lower noise levels than dominant features (large shapes, backgrounds).

A concrete follow-up experiment: **implement a spatially-adaptive noise schedule** where the beta_t values vary per image region based on local frequency content (high-frequency regions get a slower schedule to preserve details). Specifically:
1. Compute a frequency saliency map using the 2D Fourier transform of x_0
2. Modulate the local noise schedule: high-saliency regions receive lower β_t, slowing their destruction
3. Train and compare FID/LPIPS on CelebA-HQ vs. the standard linear schedule

This connects to my interest in privacy-preserving machine learning: a spatially-controlled schedule could also enable differential privacy in diffusion — adding more noise to sensitive facial regions (eyes, identifying features) while preserving general structure, analogous to how we apply varying epsilon budgets to different data features in federated learning.

In [ ]:
#Final summary figure: Forward process + noise schedule side by side
print('All experiments complete.')
print('\nFigures generated:')
print('  vi_results.png         — Value function heatmap + policy arrows')
print('  vi_convergence.png     — Value iteration convergence curve')
print('  gamma_experiment.png   — Policy grids across 5 discount factors')
print('  success_vs_gamma.png   — Success rate vs gamma bar chart')
print('  forward_diffusion.png  — Forward noise process on cat image')
print('  noise_schedule.png     — Signal/noise power + SNR analysis')
print('  scheduler_comparison.png — DDPM vs DDIM sample quality grid')
print('  timing_comparison.png  — Inference time bar chart')

---
## References

1. Ho, J., Jain, A., & Abbeel, P. (2020). Denoising Diffusion Probabilistic Models. *NeurIPS 2020*. https://arxiv.org/abs/2006.11239
2. Song, J., Meng, C., & Ermon, S. (2021). Denoising Diffusion Implicit Models. *ICLR 2021*. https://arxiv.org/abs/2010.02502
3. Nichol, A., & Dhariwal, P. (2021). Improved Denoising Diffusion Probabilistic Models. *ICML 2021*. https://arxiv.org/abs/2102.09672
4. Sutton, R. S., & Barto, A. G. (2018). *Reinforcement Learning: An Introduction* (2nd ed.). MIT Press. http://incompleteideas.net/book/RLbook2020.pdf
5. HuggingFace. (2024). Diffusers documentation. https://huggingface.co/docs/diffusers
6. Farama Foundation. (2024). Gymnasium documentation. https://gymnasium.farama.org/
7. Google DDPM CelebA-HQ model card. https://huggingface.co/google/ddpm-celebahq-256
8. Carlini, N. et al. (2023). Extracting Training Data from Diffusion Models. *USENIX Security 2023*. https://arxiv.org/abs/2301.13188